# 💻 Laptop Recommendation System
### ML-Based System inspired by Fashion Recommender (ResNet50 + KNN)

**Architecture:**
- 📦 Dataset: 50+ laptops with real-world specs (RAM, Processor, GPU, Price, etc.)
- 🤖 ML Models: KNN + TF-IDF Vectorizer + KMeans Clustering + PCA
- 🎯 Recommendation: Spec-based similarity using Cosine Distance + Euclidean KNN
- 🖥️ UI: Gradio Interface (runs directly inside Google Colab)

**How to run:** Click `Runtime → Run All` or press `Ctrl+F9`

## Step 1: Install & Import Libraries

In [ ]:
!pip install gradio scikit-learn pandas numpy matplotlib seaborn -q
print('All libraries installed successfully!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# ML Models
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

print('All imports successful!')

## Step 2: Build the Dataset (50 Real Laptops)

In [ ]:
data = {
    'Name': [
        'Dell XPS 15', 'MacBook Pro 14 M3', 'HP Pavilion 15', 'Lenovo IdeaPad 3',
        'ASUS ROG Strix G15', 'Acer Aspire 5', 'MacBook Air M2', 'Dell Inspiron 15',
        'Lenovo ThinkPad X1 Carbon', 'HP Spectre x360', 'MSI Raider GE76',
        'Acer Predator Helios 300', 'ASUS ZenBook 14', 'Lenovo Legion 5 Pro',
        'Dell G15 Gaming', 'HP Omen 16', 'Microsoft Surface Laptop 5',
        'Razer Blade 15', 'Samsung Galaxy Book3 Pro', 'Gigabyte AORUS 15',
        'HP EliteBook 840', 'Lenovo Yoga 9i', 'ASUS VivoBook 15', 'Acer Swift 3',
        'Dell Latitude 5530', 'MSI Modern 14', 'Apple MacBook Pro 16 M3 Pro',
        'HP Victus 15', 'Lenovo IdeaPad Gaming 3', 'ASUS TUF Gaming F15',
        'Acer Nitro 5', 'Dell Vostro 15', 'Lenovo ThinkBook 14',
        'HP Laptop 15s', 'ASUS ExpertBook B1', 'Huawei MateBook D15',
        'Honor MagicBook 16', 'Infinix InBook X2', 'Tecno MegaBook T1',
        'HP 255 G9', 'Dell Precision 5570', 'Lenovo ThinkPad T14s',
        'ASUS ProArt Studiobook', 'MSI Creator Z16', 'HP ZBook Fury 16',
        'Apple MacBook Pro 13 M2', 'Lenovo Slim 5i Pro', 'Acer Chromebook Spin 714',
        'Dell ChromeBook 3110', 'Lenovo IdeaPad Slim 3'
    ],
    'Brand': [
        'Dell','Apple','HP','Lenovo','ASUS','Acer','Apple','Dell','Lenovo','HP',
        'MSI','Acer','ASUS','Lenovo','Dell','HP','Microsoft','Razer','Samsung','Gigabyte',
        'HP','Lenovo','ASUS','Acer','Dell','MSI','Apple','HP','Lenovo','ASUS',
        'Acer','Dell','Lenovo','HP','ASUS','Huawei','Honor','Infinix','Tecno','HP',
        'Dell','Lenovo','ASUS','MSI','HP','Apple','Lenovo','Acer','Dell','Lenovo'
    ],
    'Processor': [
        'Intel i7-13700H','Apple M3','Intel i5-1235U','Intel i5-1235U',
        'AMD Ryzen 9 6900HX','Intel i5-1235U','Apple M2','Intel i5-1235U',
        'Intel i7-1265U','Intel i7-1255U','Intel i9-12900HK',
        'Intel i7-12700H','Intel i7-1260P','AMD Ryzen 7 6800H',
        'Intel i7-12700H','Intel i7-12700H','Intel i5-1235U',
        'Intel i7-12800H','Intel i7-1360P','Intel i9-12900H',
        'Intel i5-1245U','Intel i7-1260P','Intel i5-1235U','Intel i5-1235U',
        'Intel i5-1245U','Intel i7-1255U','Apple M3 Pro',
        'Intel i5-12450H','AMD Ryzen 5 6600H','Intel i7-12700H',
        'AMD Ryzen 5 6600H','Intel i5-1235U','Intel i5-1235U',
        'AMD Ryzen 5 5500U','Intel i3-1115G4','AMD Ryzen 5 5500U',
        'AMD Ryzen 5 5600H','Intel i3-1005G1','Intel i5-1235U',
        'AMD Ryzen 5 5500U','Intel i7-12700H','AMD Ryzen 7 Pro 6850U',
        'Intel i9-13980HX','Intel i9-12900H','Intel i9-12950HX',
        'Apple M2','Intel i5-1240P','Intel i5-1235U',
        'Intel Celeron N4500','Intel i5-1235U'
    ],
    'RAM_GB': [
        32,16,8,8,16,8,8,8,16,16,32,16,16,16,8,16,8,16,16,32,
        16,16,8,8,8,16,36,8,8,16,8,8,8,8,4,8,16,8,8,4,
        16,16,32,32,32,8,16,8,4,8
    ],
    'Storage_GB': [
        512,512,256,256,512,256,256,512,512,512,1000,512,512,512,
        512,512,256,512,512,1000,512,512,256,256,256,512,512,
        512,256,512,512,256,256,256,128,256,512,256,256,256,
        512,512,1000,1000,2000,256,512,128,64,256
    ],
    'Storage_Type': [
        'SSD','SSD','SSD','HDD','SSD','HDD','SSD','HDD','SSD','SSD','SSD',
        'SSD','SSD','SSD','SSD','SSD','SSD','SSD','SSD','SSD',
        'SSD','SSD','HDD','SSD','SSD','SSD','SSD','SSD','SSD','SSD',
        'SSD','HDD','SSD','HDD','SSD','SSD','SSD','SSD','SSD','HDD',
        'SSD','SSD','SSD','SSD','SSD','SSD','SSD','SSD','eMMC','SSD'
    ],
    'GPU': [
        'NVIDIA RTX 4060','Apple M3 GPU','Intel Iris Xe','Intel Iris Xe',
        'NVIDIA RTX 3070 Ti','Intel Iris Xe','Apple M2 GPU','Intel Iris Xe',
        'Intel Iris Xe','Intel Iris Xe','NVIDIA RTX 3080 Ti',
        'NVIDIA RTX 3060','Intel Iris Xe','NVIDIA RTX 3070',
        'NVIDIA RTX 3060','NVIDIA RTX 3060','Intel Iris Xe',
        'NVIDIA RTX 3080','Intel Iris Xe','NVIDIA RTX 3080',
        'Intel Iris Xe','Intel Iris Xe','Intel Iris Xe','Intel Iris Xe',
        'Intel Iris Xe','Intel Iris Xe','Apple M3 Pro GPU',
        'NVIDIA RTX 2050','NVIDIA RTX 3050','NVIDIA RTX 3060',
        'NVIDIA RTX 3050','Intel Iris Xe','Intel Iris Xe',
        'AMD Radeon','Intel UHD','AMD Radeon','AMD Radeon',
        'Intel UHD','Intel UHD','AMD Radeon',
        'NVIDIA RTX A2000','AMD Radeon','NVIDIA RTX 4090',
        'NVIDIA RTX 3060 Ti','NVIDIA RTX A5000','Apple M2 GPU',
        'Intel Iris Xe','Intel Iris Xe','Intel UHD','Intel Iris Xe'
    ],
    'Display_Inch': [
        15.6,14.2,15.6,15.6,15.6,15.6,13.6,15.6,14.0,13.5,17.3,
        15.6,14.0,16.0,15.6,16.1,13.5,15.6,14.0,15.6,
        14.0,14.0,15.6,14.0,15.6,14.0,16.2,15.6,15.6,15.6,
        15.6,15.6,14.0,15.6,15.6,15.6,16.0,14.0,15.6,15.6,
        15.6,14.0,16.0,16.0,16.0,13.3,14.0,14.0,11.6,15.6
    ],
    'Battery_Hours': [
        6,18,8,7,5,8,18,7,15,13,4,5,12,6,5,5,15,5,12,4,
        12,14,8,10,10,10,22,6,6,5,5,7,10,8,10,10,8,8,8,6,
        8,15,5,4,4,18,11,8,10,8
    ],
    'Weight_KG': [
        1.86,1.60,1.75,1.65,2.30,1.70,1.24,1.85,1.12,1.36,2.90,
        2.20,1.39,2.40,2.50,2.35,1.29,2.10,1.17,2.40,
        1.40,1.37,1.80,1.45,1.75,1.40,2.15,2.40,2.10,2.30,
        2.30,1.85,1.42,1.78,1.45,1.58,1.90,1.52,1.95,1.78,
        1.77,1.41,1.85,2.10,2.99,1.29,1.56,1.45,1.24,1.73
    ],
    'Price_USD': [
        1800,1999,650,450,1500,550,1099,700,1700,1600,2500,
        1200,900,1400,950,1300,1300,3000,1400,2200,
        1100,1500,500,600,900,700,2499,750,700,900,
        700,550,650,400,350,500,700,400,450,350,
        2800,1800,3500,3000,4000,1299,900,600,250,400
    ],
    'Use_Case': [
        'Professional','Professional','Student','Student',
        'Gaming','Student','Professional','Student',
        'Business','Business','Gaming',
        'Gaming','Professional','Gaming',
        'Gaming','Gaming','Business',
        'Gaming','Professional','Gaming',
        'Business','Professional','Student','Student',
        'Business','Professional','Professional',
        'Gaming','Gaming','Gaming',
        'Gaming','Student','Business',
        'Student','Business','Student',
        'Student','Budget','Budget','Budget',
        'Professional','Business','Creative',
        'Creative','Creative','Professional',
        'Professional','Education','Education','Student'
    ]
}

df = pd.DataFrame(data)
print(f'Dataset ready! Total laptops: {len(df)}')
df.head(10)

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Laptop Dataset — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Price Distribution
axes[0,0].hist(df['Price_USD'], bins=15, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].set_title('Price Distribution (USD)')
axes[0,0].set_xlabel('Price ($)')
axes[0,0].set_ylabel('Count')

# RAM Distribution
ram_counts = df['RAM_GB'].value_counts().sort_index()
axes[0,1].bar(ram_counts.index.astype(str), ram_counts.values, color='coral', edgecolor='white')
axes[0,1].set_title('RAM Distribution (GB)')
axes[0,1].set_xlabel('RAM (GB)')
axes[0,1].set_ylabel('Count')

# Laptops by Brand
brand_counts = df['Brand'].value_counts()
axes[0,2].barh(brand_counts.index, brand_counts.values, color='mediumseagreen')
axes[0,2].set_title('Laptops by Brand')

# Use Case Pie
use_counts = df['Use_Case'].value_counts()
axes[1,0].pie(use_counts.values, labels=use_counts.index, autopct='%1.0f%%', startangle=90)
axes[1,0].set_title('Use Case Distribution')

# Price vs RAM scatter
use_colors = {'Gaming':'red','Professional':'blue','Student':'green',
              'Business':'orange','Creative':'purple','Budget':'gray','Education':'brown'}
for use, grp in df.groupby('Use_Case'):
    axes[1,1].scatter(grp['RAM_GB'], grp['Price_USD'],
                      label=use, color=use_colors.get(use,'black'), alpha=0.7, s=60)
axes[1,1].set_title('Price vs RAM by Use Case')
axes[1,1].set_xlabel('RAM (GB)')
axes[1,1].set_ylabel('Price ($)')
axes[1,1].legend(fontsize=7)

# Battery vs Weight
sc = axes[1,2].scatter(df['Weight_KG'], df['Battery_Hours'],
                        c=df['Price_USD'], cmap='viridis', s=60, alpha=0.8)
plt.colorbar(sc, ax=axes[1,2], label='Price ($)')
axes[1,2].set_title('Battery Life vs Weight')
axes[1,2].set_xlabel('Weight (KG)')
axes[1,2].set_ylabel('Battery (Hours)')

plt.tight_layout()
plt.show()

## Step 4: Feature Engineering + Train All ML Models

| ML Concept | Purpose |
|---|---|
| **LabelEncoder** | Convert categorical data (SSD/HDD) to numbers |
| **MinMaxScaler** | Normalize all features to 0–1 range |
| **TF-IDF Vectorizer** | Convert processor/GPU text into feature vectors |
| **Cosine Similarity** | Measure how similar two spec profiles are |
| **KNN** | Find the nearest matching laptops in feature space |
| **KMeans Clustering** | Group laptops into categories automatically |
| **PCA** | Reduce dimensions for 2D visualization |

In [ ]:
# --- GPU Score (high-end = 4, integrated = 1) ---
def categorize_gpu(gpu):
    gpu = gpu.lower()
    if any(x in gpu for x in ['rtx 40', 'rtx 3080', 'rtx 3090', 'a5000', 'a2000', '4090']):
        return 4
    elif any(x in gpu for x in ['rtx 3070', 'rtx 3060', 'rtx 3050', 'rtx 2050', '3060 ti']):
        return 3
    elif any(x in gpu for x in ['m3', 'm2']):
        return 3
    elif any(x in gpu for x in ['iris', 'radeon', 'uhd']):
        return 1
    return 2

# --- Processor Score (i9/Ryzen9 = 5, Celeron = 1) ---
def categorize_processor(proc):
    proc = proc.lower()
    if any(x in proc for x in ['i9', 'ryzen 9', 'm3 pro']):
        return 5
    elif any(x in proc for x in ['i7', 'ryzen 7', 'm3', 'm2']):
        return 4
    elif any(x in proc for x in ['i5', 'ryzen 5']):
        return 3
    elif any(x in proc for x in ['i3', 'ryzen 3']):
        return 2
    return 1

df['GPU_Score']        = df['GPU'].apply(categorize_gpu)
df['Processor_Score']  = df['Processor'].apply(categorize_processor)
df['Storage_Encoded']  = LabelEncoder().fit_transform(df['Storage_Type'])

# --- MODEL 1: TF-IDF on text specs ---
df['Specs_Text'] = df['Processor'] + ' ' + df['GPU'] + ' ' + df['Use_Case']
tfidf        = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['Specs_Text'])
print(f'TF-IDF Matrix: {tfidf_matrix.shape}')

# --- MODEL 2: KNN on numerical features ---
NUM_FEATURES = ['RAM_GB','Storage_GB','GPU_Score','Processor_Score',
                'Display_Inch','Battery_Hours','Weight_KG','Price_USD','Storage_Encoded']
scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(df[NUM_FEATURES])

knn_model = NearestNeighbors(n_neighbors=6, metric='euclidean', algorithm='brute')
knn_model.fit(X_scaled)
print(f'KNN trained on {len(df)} laptops with {len(NUM_FEATURES)} features')

# --- MODEL 3: KMeans Clustering ---
kmeans       = KMeans(n_clusters=6, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)
CLUSTER_NAMES = {0:'Budget',1:'Student',2:'Business',3:'Gaming',4:'Professional',5:'Creative'}
print('KMeans Clusters:', df['Cluster'].value_counts().to_dict())

# --- MODEL 4: PCA for visualization ---
pca   = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA Variance Explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

## Step 5: Visualize ML Clusters (PCA 2D Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
COLORS = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD']

# Plot 1: by KMeans Cluster
for i, (cid, label) in enumerate(CLUSTER_NAMES.items()):
    mask = df['Cluster'] == cid
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=COLORS[i], label=label, s=80, alpha=0.85)
    for idx in np.where(mask)[0]:
        axes[0].annotate(df['Name'].iloc[idx][:10],
                         (X_pca[idx,0], X_pca[idx,1]), fontsize=6, alpha=0.55)
axes[0].set_title('KMeans Clusters — PCA 2D View', fontsize=13, fontweight='bold')
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: by Use Case
use_cases  = df['Use_Case'].unique()
cmap2 = plt.cm.get_cmap('tab10', len(use_cases))
for i, uc in enumerate(use_cases):
    mask = df['Use_Case'] == uc
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[cmap2(i)], label=uc, s=80, alpha=0.85)
axes[1].set_title('Use Case Distribution — PCA 2D View', fontsize=13, fontweight='bold')
axes[1].set_xlabel('PCA Component 1')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Hybrid Recommendation Engine

In [ ]:
# Ideal text profile per use case (used by TF-IDF)
USE_CASE_PROFILES = {
    'Gaming':       'Intel i7-12700H NVIDIA RTX 3060 Gaming',
    'Student':      'Intel i5-1235U Intel Iris Xe Student',
    'Business':     'Intel i7-1265U Intel Iris Xe Business',
    'Professional': 'Intel i7-13700H NVIDIA RTX 4060 Professional',
    'Creative':     'Intel i9-12900H NVIDIA RTX 3080 Creative',
    'Budget':       'Intel i3-1115G4 Intel UHD Budget',
    'Education':    'Intel i5-1235U Intel Iris Xe Education',
}

def recommend(use_case, budget_min, budget_max, ram_min,
              storage_min, need_gpu, battery_min, weight_max):
    """
    Hybrid Recommendation:
      1. Hard filter by user constraints
      2. TF-IDF cosine similarity on text specs  (60% weight)
      3. KNN euclidean distance on numerical specs (40% weight)
      4. Combine scores and return Top 5
    """
    filtered = df[
        (df['Price_USD']     >= budget_min)  &
        (df['Price_USD']     <= budget_max)  &
        (df['RAM_GB']        >= ram_min)     &
        (df['Storage_GB']    >= storage_min) &
        (df['Battery_Hours'] >= battery_min) &
        (df['Weight_KG']     <= weight_max)
    ].copy()

    if need_gpu:
        filtered = filtered[filtered['GPU_Score'] >= 3]

    if filtered.empty:
        return pd.DataFrame({'Result': ['No laptops found. Try relaxing your filters.']})

    idx = filtered.index.tolist()

    # TF-IDF cosine similarity
    ideal_text = USE_CASE_PROFILES.get(use_case, USE_CASE_PROFILES['Student'])
    ideal_vec  = tfidf.transform([ideal_text])
    cos_scores = cosine_similarity(ideal_vec, tfidf_matrix[idx]).flatten()

    # KNN euclidean score (converted to similarity)
    ideal_num = np.array([[
        ram_min, storage_min,
        3 if need_gpu else 1, 3,
        15.0, battery_min,
        weight_max / 2,
        (budget_min + budget_max) / 2,
        1
    ]])
    ideal_scaled = scaler.transform(ideal_num)
    dists        = euclidean_distances(ideal_scaled, X_scaled[idx]).flatten()
    knn_scores   = 1 / (1 + dists)

    # Hybrid score
    hybrid = 0.6 * cos_scores + 0.4 * knn_scores
    top5   = np.argsort(hybrid)[::-1][:5]

    result = filtered.iloc[top5][[
        'Name','Brand','Processor','RAM_GB','Storage_GB',
        'GPU','Battery_Hours','Weight_KG','Price_USD','Use_Case'
    ]].copy()
    result['Match %'] = (hybrid[top5] * 100).round(1)
    result = result.rename(columns={
        'RAM_GB':'RAM (GB)', 'Storage_GB':'Storage (GB)',
        'Battery_Hours':'Battery (hrs)', 'Weight_KG':'Weight (kg)',
        'Price_USD':'Price ($)'
    })
    result.index = range(1, len(result)+1)
    return result

# Quick test
print('--- Gaming Laptop Test ---')
recommend('Gaming', 800, 1600, 16, 512, True, 5, 3.0)

## Step 7: Launch Gradio UI

In [ ]:
def gradio_fn(use_case, budget_min, budget_max, ram, storage,
               need_gpu, battery, weight):
    return recommend(use_case, budget_min, budget_max,
                     ram, storage, need_gpu, battery, weight)

with gr.Blocks(theme=gr.themes.Soft(), title='Laptop Recommender') as demo:

    gr.Markdown("""
    # 💻 AI-Powered Laptop Recommendation System
    **ML Models:** KNN + TF-IDF + Cosine Similarity + KMeans Clustering
    Fill in your requirements and click **Find Laptops**!
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Your Requirements')

            use_case = gr.Dropdown(
                choices=['Student','Gaming','Business','Professional','Creative','Budget','Education'],
                value='Student',
                label='Use Case'
            )
            with gr.Row():
                bmin = gr.Slider(100, 3000, value=400,  step=50, label='Min Budget ($)')
                bmax = gr.Slider(200, 5000, value=1000, step=50, label='Max Budget ($)')

            ram     = gr.Radio([4, 8, 16, 32],   value=8,   label='Minimum RAM (GB)')
            storage = gr.Radio([128, 256, 512, 1000], value=256, label='Minimum Storage (GB)')
            gpu_cb  = gr.Checkbox(value=False, label='Need Dedicated GPU? (Gaming / Video Editing)')
            battery = gr.Slider(2, 20, value=6, step=1,   label='Minimum Battery Life (Hours)')
            weight  = gr.Slider(1.0, 4.0, value=2.5, step=0.1, label='Maximum Weight (KG)')

            btn = gr.Button('Find Laptops', variant='primary', size='lg')

        with gr.Column(scale=2):
            gr.Markdown('### Top 5 Recommended Laptops')
            output = gr.Dataframe(wrap=True)

    gr.Markdown("""
    ---
    ### How the ML Engine Works
    | Step | Model | What it does |
    |------|-------|--------------|
    | 1 | **Hard Filter** | Removes laptops outside your budget / RAM / weight |
    | 2 | **TF-IDF** | Converts processor & GPU names into text vectors |
    | 3 | **Cosine Similarity** | Scores how well each laptop's text profile matches your use case |
    | 4 | **KNN + Euclidean** | Scores similarity on numerical specs (price, RAM, battery…) |
    | 5 | **Hybrid Score** | 60% text score + 40% numerical score → ranked Top 5 |
    """)

    gr.Examples(
        examples=[
            ['Student',      300,  600,  8,  256, False, 8,  2.0],
            ['Gaming',       900, 1600, 16,  512, True,  5,  3.0],
            ['Professional',1000, 2000, 16,  512, False, 10, 2.0],
            ['Budget',       200,  450,  4,  128, False,  6, 2.5],
            ['Creative',    1500, 4000, 32,  512, True,   4, 2.5],
        ],
        inputs=[use_case, bmin, bmax, ram, storage, gpu_cb, battery, weight],
        label='Quick Examples — Click to try!'
    )

    btn.click(
        fn=gradio_fn,
        inputs=[use_case, bmin, bmax, ram, storage, gpu_cb, battery, weight],
        outputs=output
    )

# share=True gives you a public URL that works in Colab
demo.launch(share=True)

## Step 8: Feature Correlation Heatmap + Cluster Price Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Correlation Heatmap
num_cols = ['RAM_GB','Storage_GB','GPU_Score','Processor_Score',
            'Battery_Hours','Weight_KG','Price_USD']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')

# Average Price per Cluster
cluster_stats = df.groupby('Cluster')['Price_USD'].mean()
names  = [CLUSTER_NAMES.get(i, f'C{i}') for i in cluster_stats.index]
colors = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD']
bars   = axes[1].bar(names, cluster_stats.values, color=colors)
for bar, val in zip(bars, cluster_stats.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'${val:.0f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Average Price by Cluster', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Average Price ($)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Project Summary

```
Fashion Recommender  →  Laptop Recommender
─────────────────────────────────────────────
ResNet50             →  TF-IDF Vectorizer
Image embeddings     →  Spec vectors
KNN on images        →  KNN on numerical specs
Euclidean distance   →  Euclidean + Cosine
Streamlit UI         →  Gradio UI (Colab ready)
─────────────────────────────────────────────

ML Models:
  KNN (K-Nearest Neighbors)
  TF-IDF Vectorizer
  Cosine Similarity
  KMeans Clustering
  PCA (Dimensionality Reduction)
  MinMaxScaler (Normalization)
  LabelEncoder
  Hybrid Scoring (60% text + 40% numerical)
```